<a href="https://colab.research.google.com/github/Sattvikannadatha/LLM-Ticket/blob/main/Advanced_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install dependencies
!pip install -q fastapi uvicorn streamlit pandas numpy langchain-google-genai langchain-experimental python-docx

import os, zipfile, shutil
from google.colab import files

# 2. Check for local CSV or extract from zip
def prepare_data():
    if os.path.exists('/content/support_tickets.csv'):
        print("✅ support_tickets.csv is ready.")
        return

    # Check for zip archives
    zip_candidates = ['/content/AI Assessment.zip', '/content/drive/MyDrive/AI Assessment.zip']
    for zpath in zip_candidates:
        if os.path.exists(zpath):
            with zipfile.ZipFile(zpath, 'r') as zip_ref:
                zip_ref.extractall('/content/ai_assessment_extracted')
            if os.path.exists('/content/ai_assessment_extracted/support_tickets.csv'):
                shutil.copy('/content/ai_assessment_extracted/support_tickets.csv', '/content/support_tickets.csv')
                print("✅ Extracted support_tickets.csv from ZIP.")
                return

    # If missing everywhere, prompt direct upload
    print("📁 Please upload 'support_tickets.csv' or 'AI Assessment.zip':")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/ai_assessment_extracted')
            if os.path.exists('/content/ai_assessment_extracted/support_tickets.csv'):
                shutil.copy('/content/ai_assessment_extracted/support_tickets.csv', '/content/support_tickets.csv')
        elif filename == 'support_tickets.csv':
            shutil.copy(filename, '/content/support_tickets.csv')

    if os.path.exists('/content/support_tickets.csv'):
        print("✅ Data file uploaded and ready!")
    else:
        print("❌ Upload failed. Please try running this cell again.")

prepare_data()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

In [2]:
api_code = """
from fastapi import FastAPI
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

app = FastAPI()
DATA_PATH = 'support_tickets.csv'

def load_df():
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
        return df
    return pd.DataFrame()

@app.get('/health')
def health():
    return {'status': 'healthy'}

@app.get('/anomalies')
def get_anomalies():
    df = load_df()
    if df.empty: return []
    now = df['created_at'].max()

    # Rule: Unresolved High/Critical priority tickets older than 24h
    unresolved = df[(df['status'] != 'Resolved') &
                    (df['priority'].isin(['High', 'Critical'])) &
                    (df['created_at'] < (now - timedelta(hours=24)))]

    return unresolved.replace({np.nan: None}).to_dict(orient='records')

@app.get('/query')
def query_data(q: str):
    return {'query': q, 'answer': 'Backend received query successfully.'}
"""

with open('main.py', 'w') as f:
    f.write(api_code.strip())
print("✅ Backend API (main.py) generated.")

✅ Backend API (main.py) generated.


In [3]:
app_code = """
import streamlit as st
import pandas as pd
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents import create_pandas_dataframe_agent

st.set_page_config(page_title='SupportPulse AI', page_icon='📈', layout='wide')
DATA_PATH = 'support_tickets.csv'

@st.cache_data
def load_data():
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
        return df
    return None

st.title('📈 SupportPulse AI | Intelligence Dashboard')
df = load_data()

if df is not None:
    tab1, tab2 = st.tabs(['📊 Operations Analytics', '🤖 AI Reasoning Engine'])

    with tab1:
        st.subheader('System Overview')
        col1, col2, col3 = st.columns(3)
        col1.metric("Total Tickets Managed", len(df))
        col2.metric("Open Tickets", len(df[df['status'] == 'Open']))
        col3.metric("Resolved Tickets", len(df[df['status'] == 'Resolved']))
        st.dataframe(df.head(20), use_container_width=True)

    with tab2:
        st.subheader('Neural Query Interface')
        user_query = st.text_input('Ask SupportPulse anything about your data (e.g., "Which agent resolved the most tickets?"):')

        if user_query:
            api_key = os.environ.get('GOOGLE_API_KEY')
            if not api_key:
                st.error('Google API Key not found in environment. Make sure GOOGLE_API_KEY is set in Colab Secrets.')
            else:
                try:
                    llm = ChatGoogleGenerativeAI(
                        model='gemini-3.5-flash-lite',
                        google_api_key=api_key,
                        temperature=0,
                        max_retries=3
                    )

                    agent = create_pandas_dataframe_agent(
                        llm,
                        df,
                        allow_dangerous_code=True,
                        agent_type="tool-calling",
                        verbose=False
                    )

                    with st.spinner("Analyzing ticket data..."):
                        response = agent.invoke({'input': user_query})
                        st.success(response['output'])

                except Exception as e:
                    st.error(f'System Error: {e}')
else:
    st.error('Missing support_tickets.csv data file. Please ensure it is uploaded.')
"""

with open('app.py', 'w') as f:
    f.write(app_code.strip())

print("✅ Streamlit UI (app.py) generated.")

✅ Streamlit UI (app.py) generated.


In [4]:
import os
import time
import subprocess
from google.colab import userdata

# 1. Clear active processes
os.system('pkill -f streamlit')
os.system('pkill -f cloudflared')
os.system('pkill -f uvicorn')
time.sleep(2)

# 2. Inject API key into system environment
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("✅ Google API Key securely configured.")
except Exception:
    print("❌ ERROR: 'GOOGLE_API_KEY' missing from Colab Secrets! Add it via the 🔑 sidebar tab.")

# 3. Launch FastAPI & Streamlit background servers
subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
print("🚀 Starting Streamlit server...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1"])
time.sleep(5)

# 4. Start Cloudflare Tunnel
print("🌍 Launching Cloudflare Tunnel...")
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
os.system('chmod +x cloudflared')
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"], stdout=open("tunnel_log.txt", "w"), stderr=subprocess.STDOUT)
time.sleep(6)

# 5. Output Public URL
url = ""
if os.path.exists("tunnel_log.txt"):
    with open("tunnel_log.txt", "r") as f:
        for line in f:
            if "trycloudflare.com" in line:
                url = [word for word in line.split() if "trycloudflare.com" in word][0]
                break

if url:
    print("\n" + "="*60)
    print("✅ YOUR APP IS LIVE ON THE INTERNET:")
    print(f"🔗 {url}")
    print("="*60)
else:
    print("\n❌ Tunnel URL generation failed. Checking logs:")
    os.system('cat tunnel_log.txt')

✅ Google API Key securely configured.
🚀 Starting Streamlit server...
🌍 Launching Cloudflare Tunnel...

✅ YOUR APP IS LIVE ON THE INTERNET:
🔗 trycloudflare.com...


In [5]:
!grep -o 'https://.*trycloudflare.com' tunnel_log.txt

https://accurately-short-negotiations-knights.trycloudflare.com


In [6]:
import pandas as pd
from datetime import timedelta
from IPython.display import display

def detect_anomalies(file_path):
    # 1. Load and prepare the data
    try:
        df = pd.read_csv(file_path)
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

    # Determine 'now' relative to the latest ticket in the dataset
    now = df['created_at'].max()

    # 2. Rule A: Unresolved High/Critical tickets older than 24 hours
    unresolved_high_priority = df[
        (df['status'] != 'Resolved') &
        (df['priority'].isin(['High', 'Critical'])) &
        (df['created_at'] < (now - timedelta(hours=24)))
    ].copy()
    unresolved_high_priority['anomaly_reason'] = 'Overdue High Priority'

    # 3. Rule B: Abnormally long resolution times (Top 5% of resolved tickets)
    res_threshold = df['resolution_time_hrs'].quantile(0.95)
    long_resolution = df[
        (df['status'] == 'Resolved') &
        (df['resolution_time_hrs'] > res_threshold)
    ].copy()
    long_resolution['anomaly_reason'] = 'Abnormally Long Resolution'

    # 4. Combine and deduplicate
    anomalies = pd.concat([unresolved_high_priority, long_resolution]).drop_duplicates(subset=['ticket_id'])

    return anomalies

# --- Execution ---
csv_path = 'support_tickets.csv' # Assumes the file is in your current working directory
anomalies_df = detect_anomalies(csv_path)

if anomalies_df is not None and not anomalies_df.empty:
    print(f"Total anomalies detected: {len(anomalies_df)}\n")

    print("--- Breakdown by Anomaly Reason ---")
    print(anomalies_df['anomaly_reason'].value_counts().to_string())

    print("\n--- Displaying Anomaly Rows ---")
    # Displays an interactive and highly readable table in Colab
    display(anomalies_df)
elif anomalies_df is not None and anomalies_df.empty:
    print("No anomalies detected based on the current thresholds.")

Total anomalies detected: 97

--- Breakdown by Anomaly Reason ---
anomaly_reason
Overdue High Priority         80
Abnormally Long Resolution    17

--- Displaying Anomaly Rows ---


,ticket_id,created_at,category,priority,status,response_time_hrs,resolution_time_hrs,agent_id,customer_rating,issue_summary,anomaly_reason
6,TKT-007,2024-01-21 15:24:00,General,High,Open,4.9,NaN,AGT-05,NaN,Question about usage limits,Overdue High Priority
12,TKT-013,2024-03-05 09:55:00,General,High,Open,1.6,NaN,AGT-11,NaN,Question about usage limits,Overdue High Priority
21,TKT-022,2024-03-11 09:03:00,Technical,High,Open,5.0,NaN,AGT-11,NaN,API timeout errors in production,Overdue High Priority
43,TKT-044,2024-02-08 12:13:00,General,High,Open,4.0,NaN,AGT-07,NaN,Request for account deletion,Overdue High Priority
57,TKT-058,2024-03-01 08:47:00,Technical,High,Escalated,0.5,NaN,AGT-09,NaN,Integration broken after version upgrade,Overdue High Priority
...,...,...,...,...,...,...,...,...,...,...,...
399,TKT-400,2024-03-28 09:11:00,Technical,Medium,Resolved,3.7,68.1,AGT-03,3.0,Dashboard not loading,Abnormally Long Resolution
408,TKT-409,2024-02-03 14:24:00,General,High,Resolved,3.2,96.5,AGT-08,4.0,How to export data to CSV,Abnormally Long Resolution
445,TKT-446,2024-02-05 11:38:00,Technical,Critical,Resolved,2.4,60.6,AGT-04,4.0,Database connection timeout,Abnormally Long Resolution
465,TKT-466,2024-02-16 17:33:00,Technical,High,Resolved,2.2,100.4,AGT-08,4.0,API timeout errors in production,Abnormally Long Resolution
